Модель цены от времени

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import t
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge

# ── ДАННЫЕ ────────────────────────────────────────────────────────────────────

df_model = df.sort_values('DATE_').reset_index(drop=True)

feature_columns = ['month', 'year', 'day', 'dayofweek', 'quarter']
target_column   = 'UNITPRICE'

X = df_model[feature_columns]
y = df_model[target_column]

# ── SCALE + POLY ──────────────────────────────────────────────────────────────

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pmf = PolynomialFeatures(degree=2, include_bias=False)
X_poly = pmf.fit_transform(X_scaled)

feature_names = pmf.get_feature_names_out(feature_columns)

# ── RIDGE MODEL ───────────────────────────────────────────────────────────────

ridge = Ridge()

ridge.fit(X_poly, y)

y_pred_all = ridge.predict(X_poly)

print(f"R² (Price ~ Time): {ridge.score(X_poly, y):.4f}")

# ── CLASSIC PREDICTION INTERVAL ──────────────────────────────────────────────

n = len(y)
p = X_poly.shape[1]

residuals = y.values - y_pred_all

mse = np.sum(residuals**2) / (n - p)
s_err = np.sqrt(mse)

XtX_inv = np.linalg.inv(X_poly.T @ X_poly)

# leverage
h = np.sum(X_poly @ XtX_inv * X_poly, axis=1)

alpha = 0.05
t_value = t.ppf(1 - alpha / 2, df=n - p)

# prediction interval
pi = t_value * s_err * np.sqrt(1 + h)

lower_all = y_pred_all - pi
upper_all = y_pred_all + pi

# ── СОХРАНЯЕМ ─────────────────────────────────────────────────────────────────

df_model['predicted_price'] = y_pred_all
df_model['price_lower_95']  = lower_all
df_model['price_upper_95']  = upper_all

# ── METRICS ───────────────────────────────────────────────────────────────────

mean_width = (upper_all - lower_all).mean()

coverage = np.mean(
    (y.values >= lower_all) &
    (y.values <= upper_all)
) * 100

print(f"Средняя ширина 95% интервала: {mean_width:.4f}")
print(f"Покрытие:                     {coverage:.2f}%")

# ── SIGNIFICANCE TEST (OLS) ──────────────────────────────────────────────────
# p-value считаем через OLS

X_ols = pd.DataFrame(X_poly, columns=feature_names)

X_ols = sm.add_constant(X_ols)

ols_model = sm.OLS(y, X_ols).fit()

coef_table = pd.DataFrame({
    'feature': ols_model.params.index,
    'coef': ols_model.params.values,
    'p_value': ols_model.pvalues.values
})

coef_table = coef_table.sort_values('p_value')

print("\n=== Значимость коэффициентов ===")
print(coef_table)

print("\n=== Полный OLS Summary ===")
print(ols_model.summary())

# ── ВИЗУАЛИЗАЦИЯ ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---------------------------------------------------
# PRED VS FACT
# ---------------------------------------------------

ax1 = axes[0]

ax1.scatter(
    y,
    y_pred_all,
    alpha=0.5,
    s=30
)

ax1.plot(
    [y.min(), y.max()],
    [y.min(), y.max()],
    'r--',
    lw=2,
    label='Идеал'
)

ax1.set_xlabel('Фактическая цена')
ax1.set_ylabel('Предсказанная цена')

ax1.set_title('Предсказание vs Факт')

ax1.legend()
ax1.grid(True, alpha=0.3)

# ---------------------------------------------------
# TIME SERIES
# ---------------------------------------------------

ax2 = axes[1]

dates = df_model['DATE_']

ax2.plot(
    dates,
    y,
    'o',
    label='Факт',
    markersize=3,
    alpha=0.6
)

ax2.plot(
    dates,
    y_pred_all,
    '-',
    label='Предсказание',
    linewidth=2
)

ax2.fill_between(
    dates,
    lower_all,
    upper_all,
    alpha=0.25,
    label='95% Prediction Interval'
)

ax2.set_xlabel('Дата')
ax2.set_ylabel('Цена')

ax2.set_title('Цена во времени с prediction interval')

ax2.legend()
ax2.grid(True, alpha=0.3)

plt.setp(
    ax2.xaxis.get_majorticklabels(),
    rotation=45,
    ha='right'
)

plt.tight_layout()
plt.show()

Модель спроса от цены

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t

# =========================================================
# ДАННЫЕ
# =========================================================

train = data_grop[['UNITPRICE', 'month', 'dayofweek', 'day', 'year', 'quarter']]
target = data_grop['AMOUNT']

# =========================================================
# ОБУЧЕНИЕ
# =========================================================

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train)

poly = PolynomialFeatures(degree=2, include_bias=False)
train_poly = poly.fit_transform(train_scaled)

model = Ridge()
model.fit(train_poly, target)

print(model.coef_)

pred = model.predict(train_poly)

print("R2:", r2_score(target, pred))
print("MSE:", np.mean((target - pred)**2))

# =========================================================
# PREDICTION INTERVAL
# =========================================================

X = train_poly
y = target.values

n = len(y)
p = X.shape[1]

# Остатки
residuals = y - pred

# Оценка дисперсии ошибок
mse = np.sum(residuals**2) / (n - p)
s_err = np.sqrt(mse)

# leverage
XtX_inv = np.linalg.inv(X.T @ X)
h = np.sum(X @ XtX_inv * X, axis=1)

# t-value
alpha = 0.05
t_value = t.ppf(1 - alpha/2, df=n - p)

# Prediction interval
pi = t_value * s_err * np.sqrt(1 + h)

lower = pred - pi
upper = pred + pi

# =========================================================
# ГРАФИК
# =========================================================

plot_df = data_grop.copy()

plot_df['pred'] = pred
plot_df['lower'] = lower
plot_df['upper'] = upper

plot_df = plot_df.sort_values('UNITPRICE')

fig, ax = plt.subplots(figsize=(5, 5))

sns.scatterplot(
    data=plot_df,
    x='UNITPRICE',
    y='AMOUNT',
    alpha=0.5,
    label='Фактические данные',
    ax=ax
)

sns.lineplot(
    data=plot_df,
    x='UNITPRICE',
    y='pred',
    color='orange',
    linewidth=2,
    label='Прогноз',
    ax=ax
)

ax.fill_between(
    plot_df['UNITPRICE'],
    plot_df['lower'],
    plot_df['upper'],
    alpha=0.25,
    label='95% Prediction Interval'
)

ax.set_title('Зависимость количества продаж от цены')
ax.set_xlabel('Цена')
ax.set_ylabel('Количество продаж')

plt.grid(True)
plt.legend()

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# =========================================================
# ДАННЫЕ
# =========================================================

X = data_grop[['UNITPRICE', 'month', 'dayofweek', 'day', 'year', 'quarter']]
y = data_grop['AMOUNT']

# =========================================================
# SCALE + POLY
# =========================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled)

feature_names = poly.get_feature_names_out(X.columns)

X_poly_df = pd.DataFrame(X_poly, columns=feature_names)

# =========================================================
# OLS
# =========================================================

X_poly_df = sm.add_constant(X_poly_df)

model = sm.OLS(y, X_poly_df).fit()

# =========================================================
# РЕЗУЛЬТАТЫ
# =========================================================

results = pd.DataFrame({
    'feature': model.params.index,
    'coef': model.params.values,
    'p_value': model.pvalues.values
})

# сортировка по значимости
results = results.sort_values('p_value')

print(results)

# Полный отчет
a = pd.DataFrame(model.summary().tables[1])